# 0 Libraries inladen

In [313]:
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import random

pd.set_option('display.float_format', lambda x: '%.2f' % x)

env = gym.make("Blackjack-v1")

sample_size = 1000 # voor testen op 1000, voor werkelijk berekenen op 1000000
tokens = 1000 # het aantal tokens om in te zetten tijdens het spelen van blackjack

# 1 Opzetten spel

In [314]:
def blackjack(keuze_strategie, inzet): 
    obs, info = env.reset() # obs bevat de speler's som, de dealer's zichtbare kaart, en of de speler een usable ace heeft
    terminated = False # of de ronde is afgelopen
    truncated = False # of de ronde is afgebroken (bijvoorbeeld door een time limit)
    
    while not (terminated or truncated): # zolang de ronde niet is afgelopen of afgebroken
        speler_som = obs[0] # de som van de speler's kaarten
        dealer_kaarten_zichtbaar = [env.unwrapped.dealer[0]] # de zichtbare kaart van de dealer
        speler_kaarten = env.unwrapped.player # de kaarten van de speler
        
        # print(f"Speler's som: {speler_som}, Dealer's zichtbare kaart: {dealer_kaarten_zichtbaar[0]}, Speler's kaarten: {speler_kaarten}") # print de huidige situatie van de speler en dealer
        
        keuze, inzet = keuze_strategie(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet) # de strategie bepaalt of de speler 'hit' of 'stick' kiest op basis van de huidige situatie
        
        if keuze == 'hit': 
            action = 1 # actie 1 betekent 'hit' in de Blackjack omgeving
        elif keuze == 'stick':
            action = 0 # actie 0 betekent 'stick' in de Blackjack omgeving
            
        obs, reward, terminated, truncated, info = env.step(action) # voer de gekozen actie uit in de omgeving en ontvang de nieuwe observatie, beloning, en of de ronde is afgelopen of afgebroken
        
    if reward > 0:
        if speler_kaarten == [1, 10]:  # Blackjack
            winst = inzet * 2.5 # bij een Blackjack wint de speler 1.5 keer zijn inzet als winst (plus de inzet terug), wat neerkomt op een totale uitbetaling van 2.5 keer de inzet
            resultaat = 'Blackjack'
        else:
            winst = inzet * 2 # bij een normale overwinning wint de speler zijn inzet terug plus een gelijk bedrag als winst
            resultaat = 'Gewonnen'
    elif reward < 0:
        winst = -inzet # bij verlies verliest de speler zijn inzet
        resultaat = 'Verloren'
    else:
        winst = 0 # bij gelijkspel is de winst nul
        resultaat = 'Gelijk'
        
    return winst, resultaat

# 2 Regels van Blackjack

Blackjack is een kaartspel wat veel gespeeld wordt in casino's. De speler speelt niet tegen andere spelers, maar tegen de bank. Het doel van Blackjack is om zo dicht bij de 21 punten te halen zonder er overheen te gaan. Als de speler dichterbij de 21 komt dan de bank, dan wint de speler. Als de bank dichterbij komt wint de bank (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014).

Voor dit portfolio is de volledige casino werking van dit spel niet relevant, hierbij gaat het dan over het plaatsen van een inzet en deze verdubbelen bij een goede hand.

## 2.1 De kaarten delen
Zodra de speler de inzet geplaatst heeft (of in het geval van dit porfolio als het spel geïnitialiseerd wordt), worden de kaarten gedeeld. De speler krijgt twee zichtbare kaarten. De bank krijgt eveneens twee kaarten, waarvan er één zichtbaar is voor de speler (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014).

## 2.2 Het spel
Zodra de kaarten gedeeld heeft de speler twee keuzes: 'hit' en 'stick'. Als de speler kiest voor 'hit' dan krijgt de speler een nieuwe kaart, als de speler kiest voor 'stick' dan is het spel voor de speler afgelopen. In het casino zijn er ook de opties 'split', 'double down', 'surrender', echter zijn deze niet in de gymnasium library versie van Blackjack (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014).

## 2.3 De dealer
Als de speler klaar is met zijn spel, gaat de dealer zijn spel spelen. Hiervoor zijn twee vaste regels:

1. Als de bank minder dan 17 punten heeft moet er 'hit' gespeeld worden.
2. Als de bank 17 punten of meer heeft moet er 'stand' gespeeld worden.

## 2.4 Uitbetaling
Als het spel is afgelopen volgt de volgende uitbetaling (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014):

- Winst speler zonder Blackjack (exact 21 punten): 1x de inzet.
- Gelijkspel: inzet blijft staan voor de volgende ronde.
- Verlies: verlies inzet.
- Winst met Blackjack: 1.5x de inzet.

## 2.5 Waardes van de kaarten
In het Blackjack worden de volgende kaartwaardes gebruikt (*Blackjack Spelregels - Regels Voor Blackjack | TOTO*, z.d.; *Blackjack: Uitleg & Spelregels*, 2025; Spelregels.Eu, 2020; Van Geest, 2014):

- 2 t/m 10: eigen waarde.
- Aas: 1 of 11 punten (welke de som dichterbij de 21 krijgt).
- Boer, Vrouw en Heer: 10 punten.

# 3 Uitleg gekozen strategieën

# 4 Strategieën uitwerken en uitvoeren

In [315]:
def baseline_strat(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    return random.choice(['hit', 'stick']), inzet # deze strategie kiest willekeurig tussen 'hit' en 'stick', ongeacht de situatie

baseline_results = [] 
baseline_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = baseline_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(baseline_strat, inzet)
    baseline_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    baseline_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de baseline_results lijst
    
print(f"Na {sample_size} rondes met de baseline strategie heeft de speler {baseline_tokens} tokens over.")

Na 1000 rondes met de baseline strategie heeft de speler 1.1903834202594978e-29 tokens over.


In [316]:
def strat1_boekje(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    if speler_som == 21:  # Blackjack
        return 'stick', inzet 
    elif 1 not in speler_kaarten: # als de speler geen Ace heeft, dan is de strategie gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer
        if dealer_kaarten_zichtbaar[0] in [2, 3]: # als de dealer een 2 of 3 heeft, is het beter om te 'hit' als de speler's som 12 of minder is, omdat de kans groot is dat de dealer zal busten (boven 21 gaat)
            if speler_som <= 12:
                return 'hit', inzet
            else:
                return 'stick', inzet
        elif dealer_kaarten_zichtbaar[0] in [4, 5, 6]: # als de dealer een 4, 5, of 6 heeft, is het beter om te 'hit' als de speler's som 11 of minder is, omdat de kans groot is dat de dealer zal busten
            if speler_som <= 11:
                return 'hit', inzet
            else:
                return 'stick', inzet
        elif dealer_kaarten_zichtbaar[0] in [7, 8, 9, 10, 1]: # als de dealer een 7, 8, 9, 10, of Ace heeft, is het beter om te 'hit' als de speler's som 16 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if speler_som <= 16:
                return 'hit', inzet
            else:
                return 'stick', inzet
    else: # als de speler een Ace heeft, dan kan de Ace als 1 of 11 worden geteld, dus de strategie is gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer, maar met een hogere drempel voor 'hit' omdat de speler meer flexibiliteit heeft met de Ace
        if dealer_kaarten_zichtbaar[0] in [2, 3, 4, 5, 6, 7, 8]: # als de dealer een 2, 3, 4, 5, 6, 7, of 8 heeft, is het beter om te 'hit' als de speler's som (met de Ace geteld als 11) 17 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6]):
                return 'hit', inzet
            else:
                return 'stick', inzet
        elif dealer_kaarten_zichtbaar[0] in [9, 10, 1]: # als de dealer een 9, 10, of Ace heeft, is het beter om te 'hit' als de speler's som (met de Ace geteld als 11) 18 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6, 7]):
                return 'hit', inzet
            else:
                return 'stick', inzet
        
    
strat1_results = []
strat1_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat1_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat1_boekje, inzet)
    strat1_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat1_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat1_results lijst
    
print(f"Na {sample_size} rondes met de boekje strategie heeft de speler {strat1_tokens} tokens over.")

Na 1000 rondes met de boekje strategie heeft de speler 2.9376087982903056e+16 tokens over.


In [317]:
def strat2_boekje_invers(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    if speler_som == 21:  # Blackjack
        return 'stick', inzet
    elif 1 not in speler_kaarten: # als de speler geen Ace heeft, dan is de strategie gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer, maar met omgekeerde beslissingen in vergelijking met de boekje strategie
        if dealer_kaarten_zichtbaar[0] in [2, 3]: # als de dealer een 2 of 3 heeft, is het beter om te 'stick' als de speler's som 12 of minder is, omdat de kans groot is dat de dealer zal busten (boven 21 gaat)
            if speler_som <= 12:
                return 'stick', inzet
            else:
                return 'hit', inzet
        elif dealer_kaarten_zichtbaar[0] in [4, 5, 6]: # als de dealer een 4, 5, of 6 heeft, is het beter om te 'stick' als de speler's som 11 of minder is, omdat de kans groot is dat de dealer zal busten
            if speler_som <= 11:
                return 'stick', inzet
            else:
                return 'hit', inzet
        elif dealer_kaarten_zichtbaar[0] in [7, 8, 9, 10, 1]: # als de dealer een 7, 8, 9, 10, of Ace heeft, is het beter om te 'stick' als de speler's som 16 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if speler_som <= 16:
                return 'stick', inzet
            else:
                return 'hit', inzet
    else: # als de speler een Ace heeft, dan kan de Ace als 1 of 11 worden geteld, dus de strategie is gebaseerd op de som van de kaarten en de zichtbare kaart van de dealer, maar met een hogere drempel voor 'stick' omdat de speler meer flexibiliteit heeft met de Ace, en met omgekeerde beslissingen in vergelijking met de boekje strategie
        if dealer_kaarten_zichtbaar[0] in [2, 3, 4, 5, 6, 7, 8]: # als de dealer een 2, 3, 4, 5, 6, 7, of 8 heeft, is het beter om te 'stick' als de speler's som (met de Ace geteld als 11) 17 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6]):
                return 'stick', inzet
            else:
                return 'hit', inzet
        elif dealer_kaarten_zichtbaar[0] in [9, 10, 1]: # als de dealer een 9, 10, of Ace heeft, is het beter om te 'stick' als de speler's som (met de Ace geteld als 11) 18 of minder is, omdat de kans groot is dat de dealer een sterke hand heeft
            if any(kaart in speler_kaarten for kaart in [2, 3, 4, 5, 6, 7]):
                return 'stick', inzet
            else:
                return 'hit', inzet
        
    
strat2_results = []
strat2_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat2_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat2_boekje_invers, inzet)
    strat2_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat2_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat2_results lijst

print(f"Na {sample_size} rondes met de inverse strategie van het boekje heeft de speler {strat2_tokens} tokens over.")

Na 1000 rondes met de inverse strategie van het boekje heeft de speler 3.6379373916166394e-49 tokens over.


In [318]:
def strat3_hit_17(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    if speler_som == 21:  # Blackjack
        return 'stick', inzet
    elif speler_som <= 17: # deze strategie kiest altijd 'hit' als de som van de speler's kaarten 17 of minder is, ongeacht de zichtbare kaart van de dealer of de specifieke kaarten van de speler, en kiest 'stick' als de som 18 of meer is
        return 'hit', inzet
    else:
        return 'stick', inzet
        
    
strat3_results = []
strat3_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat3_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat3_hit_17, inzet)
    strat3_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat3_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat3_results lijst

print(f"Na {sample_size} rondes met de hit tot 17 strategie heeft de speler {strat3_tokens} tokens over.")

Na 1000 rondes met de hit tot 17 strategie heeft de speler 786523465.981417 tokens over.


In [319]:
def strat4_stand_12(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    if speler_som == 21:  # Blackjack
        return 'stick', inzet
    elif speler_som >= 12: # deze strategie kiest altijd 'stick' als de som van de speler's kaarten 12 of meer is, ongeacht de zichtbare kaart van de dealer of de specifieke kaarten van de speler, en kiest 'hit' als de som 11 of minder is
        return 'stick', inzet
    else:
        return 'hit', inzet
        
    
strat4_results = []
strat4_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat4_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat4_stand_12, inzet)
    strat4_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat4_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat4_results lijst

print(f"Na {sample_size} rondes met de stand op 12 strategie heeft de speler {strat4_tokens} tokens over.")

Na 1000 rondes met de stand op 12 strategie heeft de speler 38803957097773.58 tokens over.


In [320]:
def strat5_dealer_weakness(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    if speler_som == 21:  # Blackjack
        return 'stick', inzet
    elif dealer_kaarten_zichtbaar[0] in [2, 3, 4, 5, 6]: # deze strategie kiest 'hit' als de dealer een zwakke kaart heeft (2, 3, 4, 5, of 6)
        return 'stick', inzet
    elif dealer_kaarten_zichtbaar[0] in [7, 8, 9, 10, 1]: # deze strategie kiest 'hit' als de dealer een sterke kaart heeft (7, 8, 9, 10, of Ace)
        if speler_som <= 21:
            return 'hit', inzet
        else:
            return 'stick', inzet
        
    
strat5_results = []
strat5_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat5_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat5_dealer_weakness, inzet)
    strat5_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat5_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat5_results lijst

print(f"Na {sample_size} rondes met de dealer weakness strategie heeft de speler {strat5_tokens} tokens over.")

Na 1000 rondes met de dealer weakness strategie heeft de speler 2.020411673749851e-34 tokens over.


In [321]:
def strat6_always_hit(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    if speler_som != 21:  # deze strategie kiest altijd 'hit' als de som van de speler's kaarten niet 21 is, ongeacht de zichtbare kaart van de dealer of de specifieke kaarten van de speler, en kiest 'stick' alleen als de speler al een Blackjack heeft
        return 'hit', inzet
    else:
        return 'stick', inzet
        
    
strat6_results = []
strat6_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat6_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat6_always_hit, inzet)
    strat6_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat6_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat6_results lijst

print(f"Na {sample_size} rondes met de always hit strategie heeft de speler {strat6_tokens} tokens over.")

Na 1000 rondes met de always hit strategie heeft de speler 2.336089246413293e-76 tokens over.


In [322]:
def strat7_inzet_aware(speler_som, dealer_kaarten_zichtbaar, speler_kaarten, inzet):
    dealer_kaart = dealer_kaarten_zichtbaar[0] # de zichtbare kaart van de dealer
    is_soft = 1 in speler_kaarten and (speler_som + 10 <= 21) # check voor bruikbare aas
    aantal_kaarten = len(speler_kaarten) 

    if aantal_kaarten == 2 and (speler_som == 10 or speler_som == 11) and dealer_kaart <= 8: 
        nieuwe_inzet = inzet * 2 # Double Down: verdubbel inzet bij totaal van 10 or 11 tegen zwakke dealer
    else:
        nieuwe_inzet = inzet
    
    if speler_som >= 21:
        return 'stick', nieuwe_inzet # bij 21 of meer altijd sticken (busted of blackjack)

    if is_soft: 
        if speler_som <= 17: # bij een soft hand (met een bruikbare aas) is het beter om te 'hit' als de som 17 of minder is, omdat de speler nog steeds kan profiteren van de flexibiliteit van de Ace
            return 'hit', nieuwe_inzet
        else:
            return 'stick', nieuwe_inzet

    if speler_som >= 17: # bij 17 of meer altijd sticken, omdat de kans groot is dat de speler zal busten bij een 'hit'
        return 'stick', nieuwe_inzet
    elif 12 <= speler_som <= 16: # bij 12 tot 16 is het beter om te 'stick' als de dealer een zwakke kaart heeft (2 tot 6), omdat de kans groot is dat de dealer zal busten, en om te 'hit' als de dealer een sterke kaart heeft (7 tot Ace), omdat de kans groot is dat de dealer een sterke hand heeft
        if dealer_kaart <= 6:
            return 'stick', nieuwe_inzet
        else:
            return 'hit', nieuwe_inzet
    else:
        return 'hit', nieuwe_inzet
        
strat7_results = []
strat7_tokens = tokens # de begin hoeveelheid tokens die de speler heeft voor het spelen van blackjack
for _ in range(sample_size):
    inzet = strat7_tokens * 0.25 # de speler zet 50% van zijn tokens in voor elke ronde
    resultaat = blackjack(strat7_inzet_aware, inzet)
    strat7_tokens += resultaat[0] # voeg de winst of verlies van de ronde toe aan de totale tokens
    strat7_results.append(resultaat[1]) # sla het resultaat van de ronde ('Gewonnen', 'Verloren', 'Gelijk') op in de strat7_results lijst

print(f"Na {sample_size} rondes met de inzet aware strategie heeft de speler {strat7_tokens} tokens over.")

Na 1000 rondes met de inzet aware strategie heeft de speler 4.80859264331404e+23 tokens over.


# 5 Strategieën vergelijken

In [323]:
df = pd.DataFrame({
    'Strategie': ['Random (Baseline)', 'Boekje', 'Boekje Invers', 'Hit tot 17', 'Stand vanaf 12', 'Dealer Weakness', 'Always Hit', 'Inzet Aware'],
    'Blackjack (%)': [round(baseline_results.count('Blackjack') / sample_size * 100, 1), round(strat1_results.count('Blackjack') / sample_size * 100, 1), round(strat2_results.count('Blackjack') / sample_size * 100, 1), round(strat3_results.count('Blackjack') / sample_size * 100, 1), round(strat4_results.count('Blackjack') / sample_size * 100, 1), round(strat5_results.count('Blackjack') / sample_size * 100, 1), round(strat6_results.count('Blackjack') / sample_size * 100, 1), round(strat7_results.count('Blackjack') / sample_size * 100, 1)],
    'Gewonnen (%)': [round((baseline_results.count('Gewonnen') + baseline_results.count('Blackjack')) / sample_size * 100, 1), round((strat1_results.count('Gewonnen') + strat1_results.count('Blackjack')) / sample_size * 100, 1), round((strat2_results.count('Gewonnen') + strat2_results.count('Blackjack')) / sample_size * 100, 1), round((strat3_results.count('Gewonnen') + strat3_results.count('Blackjack')) / sample_size * 100, 1), round((strat4_results.count('Gewonnen') + strat4_results.count('Blackjack')) / sample_size * 100, 1), round((strat5_results.count('Gewonnen') + strat5_results.count('Blackjack')) / sample_size * 100, 1), round((strat6_results.count('Gewonnen') + strat6_results.count('Blackjack')) / sample_size * 100, 1), round((strat7_results.count('Gewonnen') + strat7_results.count('Blackjack')) / sample_size * 100, 1)],
    'Verloren (%)': [round(baseline_results.count('Verloren') / sample_size * 100, 1), round(strat1_results.count('Verloren') / sample_size * 100, 1), round(strat2_results.count('Verloren') / sample_size * 100, 1), round(strat3_results.count('Verloren') / sample_size * 100, 1), round(strat4_results.count('Verloren') / sample_size * 100, 1), round(strat5_results.count('Verloren') / sample_size * 100, 1), round(strat6_results.count('Verloren') / sample_size * 100, 1), round(strat7_results.count('Verloren') / sample_size * 100, 1)],
    'Gelijk (%)': [round(baseline_results.count('Gelijk') / sample_size * 100, 1), round(strat1_results.count('Gelijk') / sample_size * 100, 1), round(strat2_results.count('Gelijk') / sample_size * 100, 1), round(strat3_results.count('Gelijk') / sample_size * 100, 1), round(strat4_results.count('Gelijk') / sample_size * 100, 1), round(strat5_results.count('Gelijk') / sample_size * 100, 1), round(strat6_results.count('Gelijk') / sample_size * 100, 1), round(strat7_results.count('Gelijk') / sample_size * 100, 1)],
    'Overgebleven Tokens': [round(baseline_tokens, 2), round(strat1_tokens, 2), round(strat2_tokens, 2), round(strat3_tokens, 2), round(strat4_tokens, 2), round(strat5_tokens, 2), round(strat6_tokens, 2), round(strat7_tokens, 2)],
    'Tokens verlies (%)': [round((tokens - baseline_tokens) / tokens * 100, 1) if baseline_tokens < tokens else 0, round((tokens - strat1_tokens) / tokens * 100, 1) if strat1_tokens < tokens else 0, round((tokens - strat2_tokens) / tokens * 100, 1) if strat2_tokens < tokens else 0, round((tokens - strat3_tokens) / tokens * 100, 1) if strat3_tokens < tokens else 0, round((tokens - strat4_tokens) / tokens * 100, 1) if strat4_tokens < tokens else 0, round((tokens - strat5_tokens) / tokens * 100, 1) if strat5_tokens < tokens else 0, round((tokens - strat6_tokens) / tokens * 100, 1) if strat6_tokens < tokens else 0, round((tokens - strat7_tokens) / tokens * 100, 1) if strat7_tokens < tokens else 0],
    'Tokens winst (%)': [round((baseline_tokens - tokens) / tokens * 100, 1) if baseline_tokens > tokens else 0, round((strat1_tokens - tokens) / tokens * 100, 1) if strat1_tokens > tokens else 0, round((strat2_tokens - tokens) / tokens * 100, 1) if strat2_tokens > tokens else 0, round((strat3_tokens - tokens) / tokens * 100, 1) if strat3_tokens > tokens else 0, round((strat4_tokens - tokens) / tokens * 100, 1) if strat4_tokens > tokens else 0, round((strat5_tokens - tokens) / tokens * 100, 1) if strat5_tokens > tokens else 0, round((strat6_tokens - tokens) / tokens * 100, 1) if strat6_tokens > tokens else 0, round((strat7_tokens - tokens) / tokens * 100, 1) if strat7_tokens > tokens else 0]
})

In [324]:
df

,Strategie,Blackjack (%),Gewonnen (%),Verloren (%),Gelijk (%),Overgebleven Tokens,Tokens verlies (%),Tokens winst (%)
0,Random (Baseline),1.20,29.10,66.90,4.00,0.00,100.00,0.00
1,Boekje,2.00,42.80,50.10,7.10,29376087982903056.00,0.00,2937608798290205.50
2,Boekje Invers,2.10,23.80,75.30,0.90,0.00,100.00,0.00
3,Hit tot 17,1.90,39.30,51.20,9.50,786523465.98,0.00,78652246.60
4,Stand vanaf 12,2.10,41.50,50.60,7.90,38803957097773.58,0.00,3880395709677.40
5,Dealer Weakness,2.10,28.20,69.70,2.10,0.00,100.00,0.00
6,Always Hit,1.70,13.90,83.00,3.10,0.00,100.00,0.00
7,Inzet Aware,2.30,43.90,47.40,8.70,480859264331404017664000.00,0.00,48085926433140401766400.00


# Literatuurlijst

- *Blackjack spelregels - Regels voor Blackjack | TOTO.* (z.d.). Geraadpleegd op 12 maart 2026, van https://www.toto.nl/klantenservice/spelregels-voor-blackjack
- *Blackjack: uitleg & spelregels.* (2025, 4 november). Club BetCity. Geraadpleegd op 12 maart 2026, van https://club.betcity.nl/blackjack/regels
- Spelregels.Eu. (2020, 20 april). *Hoe speel je Blackjack.* spelregels.eu. Geraadpleegd op 12 maart 2026, van https://www.spelregels.eu/blackjack
- Van Geest, M. (2014). *Blackjack: regels en uitleg.* Meneer Casino. Geraadpleegd op 12 maart 2026, van https://meneercasino.com/online-casino-tips/blackjack-regels-en-uitleg